# Advanced Features & Refactoring

## What's covered

- **Data sources** — reading existing infrastructure you don't manage
- The most useful built-in data sources for any AWS configuration
- **`terraform_remote_state`** — pulling outputs from another root's state
- **Dynamic blocks** — programmatically generating nested configuration
- **Provisioners** — what they are, why you should almost never use them, and the modern alternatives
- **`moved` blocks** — safe in-place refactors that survive a fresh checkout
- **`import` blocks** (Terraform 1.5+) — declarative imports as part of the configuration
- **`precondition` / `postcondition`** — invariants enforced at plan and apply time


## Data sources — reading the world

A **data source** queries the cloud for information about a resource that Terraform does not manage. The data is available throughout your configuration as `data.<type>.<name>.<attribute>`.

Two common cases:

- **Looking up a resource someone else owns.** Your team manages the application; another team manages the VPC. You read their VPC by tag, get its ID and subnets, and wire your app into it. You never create or modify the VPC.
- **Reading values the cloud provides.** Current AWS account ID. Available availability zones in this region. The latest official AMI for Amazon Linux 2023. These aren't owned by anyone; they're inherent properties of the cloud.

```hcl
data "aws_caller_identity" "current" {}

data "aws_region" "current" {}

data "aws_availability_zones" "available" {
  state = "available"
}

data "aws_ami" "amazon_linux_2023" {
  most_recent = true
  owners      = ["amazon"]

  filter {
    name   = "name"
    values = ["al2023-ami-2023.*-x86_64"]
  }
}

resource "aws_instance" "web" {
  ami           = data.aws_ami.amazon_linux_2023.id
  instance_type = "t3.micro"

  tags = {
    AccountID = data.aws_caller_identity.current.account_id
    Region    = data.aws_region.current.name
  }
}
```

A data source has no `lifecycle` block (nothing to manage), no `count`/`for_each` at the resource level in older Terraform (newer versions allow it on data blocks), and is evaluated at **every** plan — the value is freshly fetched each time. That makes data sources excellent for "always use the latest matching AMI" patterns and a poor fit for anything that should be stable across plans.


## Data sources you'll use constantly

A short field guide of AWS data sources that show up in real configurations:

| Data source | Use |
|---|---|
| `aws_caller_identity` | The account ID and ARN running this Terraform — for tagging, IAM policy interpolation |
| `aws_region` | The region the provider is configured for |
| `aws_availability_zones` | List of AZs in the region; pair with `for_each` to span them |
| `aws_ami` | Look up an AMI by owner + name filter — pinned AMI without hardcoding the ID |
| `aws_vpc` | Read a VPC by ID, tags, or CIDR — for wiring into someone else's network |
| `aws_subnets` | Read a list of subnets matching filters |
| `aws_iam_policy_document` | Build an IAM policy as HCL instead of JSON-encoded string (much nicer) |
| `aws_iam_role` | Read an existing role's ARN |
| `aws_secretsmanager_secret_version` | Pull a secret's value into a resource — covered in notebook 03 |
| `aws_kms_alias` / `aws_kms_key` | Resolve a KMS key by alias |
| `aws_route53_zone` | Read a hosted zone by domain |
| `aws_organizations_organization` | Read org-level info |

**The `aws_iam_policy_document` data source is worth a special mention.** It lets you build IAM policy JSON from HCL blocks instead of writing JSON inline. The HCL version is type-checked, gets syntax highlighting, and is easier to review than a giant `jsonencode({...})`:

```hcl
data "aws_iam_policy_document" "s3_read" {
  statement {
    sid     = "AllowGetObject"
    actions = ["s3:GetObject"]
    resources = ["${aws_s3_bucket.assets.arn}/*"]
  }
}

resource "aws_iam_policy" "s3_read" {
  policy = data.aws_iam_policy_document.s3_read.json
}
```

The `.json` attribute serializes the result. Once you've used this pattern a few times, hand-writing IAM JSON feels primitive.


## `terraform_remote_state` — reading another root's outputs

When you split state across roots (notebook 06's component-per-environment pattern), one root often needs values from another. The `terraform_remote_state` data source reads outputs from a remote state file directly:

```hcl
data "terraform_remote_state" "network" {
  backend = "s3"
  config = {
    bucket = "myorg-terraform-state"
    key    = "live/prod/network/terraform.tfstate"
    region = "us-east-1"
  }
}

resource "aws_instance" "web" {
  subnet_id = data.terraform_remote_state.network.outputs.public_subnet_ids[0]
  vpc_id    = data.terraform_remote_state.network.outputs.vpc_id
}
```

The `outputs` attribute is a map of every `output` declared in the source state.

**The trade-offs vs first-class data sources:**

- **`terraform_remote_state`** is fast (one S3 GET) but couples you to the *other* state file's structure. Renaming an output there breaks consumers here.
- **First-class data sources** (`aws_vpc`, `aws_subnets`) query the cloud directly. Slower per plan but decoupled — the network root could be rewritten in CloudFormation tomorrow and the consumer wouldn't notice as long as the cloud objects are still tagged the same way.

The modern recommendation, especially in larger orgs: **prefer first-class data sources with stable tags** as the cross-team contract. `terraform_remote_state` is fine within one team that controls both ends and ships them together.


## Dynamic blocks — generating nested configuration

Some resources have **nested blocks** — not arguments, but sub-blocks that can repeat. The classic example is a security group's `ingress` block:

```hcl
resource "aws_security_group" "web" {
  vpc_id = var.vpc_id

  ingress {
    from_port   = 80
    to_port     = 80
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  ingress {
    from_port   = 443
    to_port     = 443
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  ingress {
    from_port       = 22
    to_port         = 22
    protocol        = "tcp"
    security_groups = [var.bastion_sg_id]
  }
}
```

What if the rules vary by environment, or are loaded from a variable? You can't put `for_each` on the block itself (that's for resources). The answer is a **dynamic block**:

```hcl
variable "ingress_rules" {
  type = list(object({
    from_port   = number
    to_port     = number
    protocol    = string
    cidr_blocks = optional(list(string))
    description = optional(string)
  }))
}

resource "aws_security_group" "web" {
  vpc_id = var.vpc_id

  dynamic "ingress" {
    for_each = var.ingress_rules
    content {
      from_port   = ingress.value.from_port
      to_port     = ingress.value.to_port
      protocol    = ingress.value.protocol
      cidr_blocks = ingress.value.cidr_blocks
      description = ingress.value.description
    }
  }
}
```

The structure:

- **`dynamic "<block_name>"`** says "I'm going to generate zero or more `<block_name>` blocks."
- **`for_each`** is what to iterate over (list, set, or map).
- **`content { ... }`** is the body of each generated block. Inside, `<block_name>.key` and `<block_name>.value` work the same as the resource-level `each.key` / `each.value`.

**When to use dynamic blocks.** Genuinely variable nested configuration — security group rules from a variable, IAM policy statements built from a list, autoscaling notifications keyed by environment.

**When not to.** When the nested blocks are static, just write them out. A dynamic block hiding three hardcoded ingress rules is harder to read than three plain blocks.


## Provisioners — and why to avoid them

A **provisioner** runs a script or command as part of resource creation or destruction. Three types:

- **`local-exec`** — runs a command on the machine running Terraform.
- **`remote-exec`** — SSHs into the created resource and runs a command on it.
- **`file`** — copies a file from the Terraform host to the created resource.

```hcl
resource "aws_instance" "web" {
  ami           = data.aws_ami.amazon_linux_2023.id
  instance_type = "t3.micro"

  provisioner "remote-exec" {
    inline = [
      "sudo dnf install -y nginx",
      "sudo systemctl start nginx",
    ]
    connection {
      type = "ssh"
      host = self.public_ip
      user = "ec2-user"
    }
  }
}
```

**HashiCorp's own documentation calls provisioners "a last resort."** Three problems:

- **They're imperative inside a declarative tool.** Terraform doesn't know what a provisioner did; it can't model the state of the running system after the command. The next plan can't detect drift from anything a provisioner changed.
- **They only run at creation.** If the script fails, Terraform marks the resource as tainted and recreates it next apply. If the script needs to run on update, it doesn't.
- **They couple Terraform's runtime to the resource's network.** `remote-exec` requires SSH from the Terraform host to the resource. That works on a laptop; it's painful in CI with private subnets and bastion hosts.

**The modern alternatives:**

- **Cloud-init / user-data.** Pass a script to the instance via the cloud's native init system. The cloud runs the script; Terraform doesn't need to reach the instance. This is the right answer for "configure the instance at boot."
- **AMI baking.** Build an AMI with [Packer](https://packer.io/), bake the configuration in, then have Terraform launch instances from that AMI. Faster boot, no runtime configuration.
- **Configuration management tools.** Ansible, Chef, Puppet for ongoing instance state. Terraform handles infrastructure; CM tools handle on-host configuration.
- **`null_resource` with `local-exec`** for orchestration tasks (a CLI call to register a webhook, an API call to trigger a downstream pipeline). This is the *one* provisioner pattern that's genuinely useful, and it should be the only one in your codebase.

If you find yourself writing a `remote-exec`, ask: can this go in user-data instead? The answer is almost always yes.


## `moved` blocks — safe refactors

Renaming or relocating a resource in HCL changes its **address**. Without telling Terraform that the new address is the same resource as the old, the plan shows the old being destroyed and the new being created — disaster for stateful resources. Notebook 03 covered `terraform state mv` as the imperative fix.

The **`moved` block** (Terraform 1.1+) is the *declarative* answer: encode the rename in the configuration itself, where it survives a fresh checkout and reads as part of the code.

```hcl
moved {
  from = aws_s3_bucket.logs
  to   = aws_s3_bucket.access_logs
}

resource "aws_s3_bucket" "access_logs" {
  bucket = "myorg-prod-access-logs"
}
```

When Terraform processes the configuration, it sees the `moved` block, looks up `aws_s3_bucket.logs` in state, and renames the address to `aws_s3_bucket.access_logs` *in state* — without touching the cloud. The next plan shows zero changes.

**`moved` blocks compose with module restructuring.** Moving a resource into a module, splitting a module, renaming a module — all handled with the right `moved` blocks:

```hcl
# Moving a resource into a child module
moved {
  from = aws_s3_bucket.logs
  to   = module.storage.aws_s3_bucket.logs
}

# Splitting one resource into a for_each instance
moved {
  from = aws_s3_bucket.logs
  to   = aws_s3_bucket.bucket["logs"]
}
```

**Lifetime.** Leave `moved` blocks in the code for at least one release cycle so every operator's state catches up. After that, you can delete them — the addresses have all converged. Some teams leave them forever as a historical record; either is fine.

**Why prefer this over `terraform state mv`.** State surgery requires the operator to remember to run the command. `moved` blocks are self-documenting and reproducible. Every fresh clone of the repo applies the rename correctly. State surgery commands now exist for one-off fixes, not for planned refactors.


## `import` blocks — declarative adoption

The `terraform import` command (notebook 03) is imperative — you run it once, state changes, you write the HCL after the fact. Terraform 1.5+ ships the **`import` block** as the declarative equivalent: describe the import in the configuration, and Terraform applies it on the next `apply`.

```hcl
import {
  to = aws_s3_bucket.legacy_logs
  id = "myorg-old-logs-bucket"
}

resource "aws_s3_bucket" "legacy_logs" {
  bucket = "myorg-old-logs-bucket"
}
```

`terraform plan` reports:

```
Plan: 1 to import, 0 to add, 0 to change, 0 to destroy.

  # aws_s3_bucket.legacy_logs will be imported
  resource "aws_s3_bucket" "legacy_logs" {
    bucket = "myorg-old-logs-bucket"
    ...
  }
```

After apply, state has the imported resource, and the import block can be removed. The HCL stays.

**Two big wins over the command:**

- **Code review covers the import.** The PR shows the `import` block alongside the resource definition. A reviewer sees both the intent ("we're adopting this existing bucket") and the resource it'll be matched against.
- **It composes with `terraform plan -out`.** The same plan-then-apply discipline applies — review the plan, apply the saved plan. No surprises.

**`for_each` on import blocks** (Terraform 1.7+) lets you import many resources at once:

```hcl
locals {
  buckets_to_import = ["old-bucket-1", "old-bucket-2", "old-bucket-3"]
}

import {
  for_each = toset(local.buckets_to_import)
  to       = aws_s3_bucket.imported[each.value]
  id       = each.value
}

resource "aws_s3_bucket" "imported" {
  for_each = toset(local.buckets_to_import)
  bucket   = each.value
}
```

For large imports — adopting all of someone's existing infrastructure — this is night-and-day better than running `terraform import` two hundred times.


## `precondition` / `postcondition` — invariants in code

The `lifecycle` block accepts two more meta-arguments for asserting invariants:

- **`precondition`** — checked *before* the resource is created/updated. Use to validate that inputs satisfy what this resource expects.
- **`postcondition`** — checked *after* the resource exists. Use to validate that the resource's state matches what we expected.

```hcl
data "aws_ami" "ubuntu" {
  most_recent = true
  owners      = ["099720109477"]

  filter {
    name   = "name"
    values = ["ubuntu/images/hvm-ssd/ubuntu-jammy-22.04-amd64-*"]
  }

  lifecycle {
    postcondition {
      condition     = self.architecture == "x86_64"
      error_message = "Resolved AMI must be x86_64; got ${self.architecture}."
    }
  }
}

resource "aws_instance" "web" {
  ami           = data.aws_ami.ubuntu.id
  instance_type = "t3.micro"

  lifecycle {
    precondition {
      condition     = !startswith(self.instance_type, "t4g.")
      error_message = "ARM instance types (t4g.*) require an ARM AMI."
    }
  }
}
```

**Compared to `variable.validation`** — variable validations run at the boundary of the configuration. Pre/postconditions run on individual resources, with access to attributes (via `self`) and to other resources' attributes. They catch things variable validation can't see: derived inputs, downstream attributes, cross-resource constraints.

**Use cases:**

- **Verify a derived input.** A subnet's `cidr_block` was carved out of a VPC's CIDR — assert it actually fits.
- **Verify a data source resolved sensibly.** The AMI lookup returned an x86_64 image, not an ARM one.
- **Catch dangerous combinations.** ARM AMI + x86 instance type.

The error messages show up in `terraform plan` output, with a clear pointer to the failing block. Better than a five-minute apply that fails halfway through with a cryptic provider error.


## `terraform_data` and the modern null_resource

The old `null_resource` was the catch-all for "I need a resource that doesn't actually create anything in the cloud, so I can attach `triggers` or `provisioners`." It still exists but is deprecated for trigger-style use.

**`terraform_data`** (Terraform 1.4+) is the modern replacement for derivation:

```hcl
resource "terraform_data" "lambda_build" {
  input = filebase64sha256("${path.module}/lambda.zip")
}

resource "aws_lambda_function" "worker" {
  function_name    = "worker"
  filename         = "lambda.zip"
  source_code_hash = filebase64sha256("${path.module}/lambda.zip")
  # ...

  lifecycle {
    replace_triggered_by = [terraform_data.lambda_build.id]
  }
}
```

When the zip changes, `terraform_data.lambda_build.input` changes, `id` changes, and the Lambda is replaced — even though the Lambda's *visible* config didn't change.

**`null_resource` is still useful** for `local-exec` orchestration:

```hcl
resource "null_resource" "register_webhook" {
  triggers = {
    endpoint = aws_lambda_function.worker.invoke_arn
  }

  provisioner "local-exec" {
    command = "./scripts/register-webhook.sh ${self.triggers.endpoint}"
  }
}
```

This is the legitimate `local-exec` pattern: a one-shot orchestration that runs when its trigger changes, doing something Terraform's providers don't cover natively. Don't reach for it casually; do reach for it when it's the right tool.


## Forward

Notebook eight closes the series with **Real-World Patterns & Troubleshooting** — a worked end-to-end AWS example (VPC + ALB + ASG + RDS, built as composed modules), how to organize state at scale (splitting by blast radius), drift detection in production, cost control through tagging discipline and estimation tools, the debugging workflow when Terraform misbehaves (`TF_LOG`, state surgery, the most common error patterns), and the honest question of when Terraform is the wrong tool entirely.
